### ingesting data from github folder to volume and storing the data in UC Table format

In [0]:
import urllib.request
import os, re
from pyspark.sql.utils import AnalysisException

In [0]:
download_files = {"carousal":("https://raw.githubusercontent.com/f3gt/cloud_engineering/main/dataset/CAROUSAL.xlsx"),
"pod":("https://raw.githubusercontent.com/f3gt/cloud_engineering/main/dataset/POD.xlsx"),
"hcs":("https://raw.githubusercontent.com/f3gt/cloud_engineering/main/dataset/HCS.xlsx")}

In [0]:
TARGET_CATALOG  = "engineering"
TARGET_SCHEMA = "default"
TARGET_VOLUME = "raw_prd_stream"
TARGET_BRONZE = "bronze"

In [0]:
## select catalog and schema to store volume data
try:
    spark.sql(f"use catalog `{TARGET_CATALOG}`")
    print(f"using catalog {TARGET_CATALOG}")
except AnalysisException as e:
    spark.sql(f"create catalog `{TARGET_CATALOG}`")
    print(f"Created catalog {TARGET_CATALOG}")

### using or creating new schema 

try:
    spark.sql(f"use schema `{TARGET_SCHEMA}`")
    print(f"using schema {TARGET_SCHEMA}")
except AnalysisException as e:
    spark.sql(f"create schema `{TARGET_SCHEMA}`")
    print(f"Created schema {TARGET_SCHEMA}")

spark.sql(f"create volume if not exists `{TARGET_CATALOG}`.`{TARGET_SCHEMA}`.`{TARGET_VOLUME}`")
print(f"using volume {TARGET_CATALOG}.{TARGET_SCHEMA}.{TARGET_VOLUME}")




In [0]:
### volume path to download data from github
volume_path = f"/Volumes/{TARGET_CATALOG}/{TARGET_SCHEMA}/{TARGET_VOLUME}/"


print(volume_path)

In [0]:
for filename,url in download_files.items():
    dest_path = volume_path+filename
    try:
        urllib.request.urlretrieve(url,dest_path)
        print(f"downloading {filename} to {dest_path}")
    except Exception as e:
        print(dest_path)
 


In [0]:
display(spark.read.format("excel")
        .option("useHeader", "true")
        .option("inferSchema", "true")
        .load("/Volumes/engineering/default/raw_prd_stream/carousal").count())


Creating clean table name

In [0]:

def clean_table_name(table_name:str):
    base = os.path.splitext(table_name)[0]
    base = re.sub(r"[^a-zA-Z0-9]", "_", base).lower()
    return base or "table_from_csv"

In [0]:
for filename,url in download_files.items():
    print(f"{clean_table_name(filename)}")
 

In [0]:
display(spark.read.excel("/Volumes/engineering/default/raw_prd_stream/carousal", headerRows=1).limit(10))

cleaning column name in uc table format and writing to uc table

In [0]:
def clean_column_name(column_name:str):
    base = os.path.splitext(column_name)[0]
    base = re.sub(r"[^a-zA-Z0-9]", "_", base).lower()
    return base or "col_from_csv"

In [0]:
for filename,url in download_files.items():
    reading_file = volume_path+filename
    cleaned_table_name = clean_table_name(filename)
    uc_table = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{cleaned_table_name}"
    print(f"created table {uc_table}")
    
    df = spark.read.excel(reading_file, headerRows=1)
    df = df.toDF(*[clean_column_name(c) for c in df.columns])

    try:
        df.write.mode("overwrite").saveAsTable(uc_table)
        print(f"writing table {uc_table}")
    except Exception as e:
        print(e)
        print(f"failed to write table {uc_table}")